In [23]:
import os
os.environ['USE_PYGEOS'] = '0'

import sys
from pathlib import Path
import geopandas as gpd

parent_dir = Path().resolve().parent
sys.path.insert(0, str(parent_dir))
from map2grid import *

In [24]:
gens_gdf = gpd.read_file('../data/osm/vietnam-latest_plants_and_generators.gpkg')
gens_gdf.head()

,osm_id,power,voltage,utility,name,frequency,plant_output_electricity,plant_source,plant_method,generator_output_electricity,generator_source,generator_type,substation,location,cables,circuits,line,layer,osm_id_right,geometry
0,2495831951,generator,NaN,NaN,Nhà máy nhiệt điện HẢi Phòng,NaN,NaN,NaN,NaN,yes,coal,francis_turbine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (106.65356 20.84855)
1,4037744488,plant,NaN,NaN,Nhà máy thủy điện Ialy,NaN,720 MW,hydro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (107.79546 14.22273)
2,4195284426,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500 kW,wind,horizontal_axis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (108.68039 11.20209)
3,4195284427,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500 kW,wind,horizontal_axis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (108.67951 11.20397)
4,4195284428,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500 kW,wind,horizontal_axis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (108.68835 11.20445)


In [25]:
# 步骤 1：新增 installed_capacity 列
def convert_to_mw(value):
    if pd.isnull(value):
        return None
    if "MW" in value:
        return float(value.split()[0])
    elif "kW" in value:
        return float(value.split()[0]) / 1000
    elif "GW" in value:
        return float(value.split()[0]) * 1000
    else:
        raise ValueError(f"Unexpected value: {value}")

In [26]:
installed_capacity = []
for i, row in gens_gdf.iterrows():
    try:
        if pd.notnull(row["plant_output_electricity"]):
            installed_capacity.append(convert_to_mw(row["plant_output_electricity"]))
        elif pd.notnull(row["generator_output_electricity"]):
            installed_capacity.append(convert_to_mw(row["generator_output_electricity"]))
        else:
            installed_capacity.append(None)
    except ValueError as e:
        print("osm_id ", row['osm_id'], ": ", e)
        installed_capacity.append(None)

gens_gdf["installed_capacity"] = installed_capacity

osm_id  2495831951 :  Unexpected value: yes
osm_id  9445033851 :  Unexpected value: yes
osm_id  9445033852 :  Unexpected value: yes
osm_id  9445033853 :  Unexpected value: yes
osm_id  9445033854 :  Unexpected value: yes
osm_id  9742040890 :  Unexpected value: yes
osm_id  9742066679 :  Unexpected value: yes
osm_id  9742066689 :  Unexpected value: yes
osm_id  9742066690 :  Unexpected value: yes
osm_id  9742100904 :  Unexpected value: yes
osm_id  9742100905 :  Unexpected value: yes
osm_id  9742100906 :  Unexpected value: yes
osm_id  9742100907 :  Unexpected value: yes
osm_id  9742100908 :  Unexpected value: yes
osm_id  9742106428 :  Unexpected value: yes
osm_id  9742121170 :  Unexpected value: yes
osm_id  9742121171 :  Unexpected value: yes
osm_id  9742473105 :  Unexpected value: yes
osm_id  9742498114 :  Unexpected value: yes
osm_id  10208684281 :  Unexpected value: yes
osm_id  10208700917 :  Unexpected value: yes
osm_id  11436663294 :  could not convert string to float: '3.x'
osm_id  11

In [36]:
# 步骤 2：新增 energy_source 列
gens_gdf["energy_source"] = gens_gdf["plant_source"].combine_first(gdf["generator_source"])

# # 保留处理后的 GeoDataFrame
# processed_gdf = gens_gdf.copy()

In [37]:
# 步骤 3：统计 energy_source 的唯一值
unique_energy_sources = gens_gdf["energy_source"].dropna().unique()
print("Unique energy sources:", unique_energy_sources)

Unique energy sources: ['coal' 'hydro' 'wind' 'biofuel' 'diesel' 'biogas' 'solar' 'gas' 'gas;oil'
 'oil' 'waste']


In [38]:
gens_gdf[gens_gdf["energy_source"].isin(["diesel"])]

,osm_id,power,voltage,utility,name,frequency,plant_output_electricity,plant_source,plant_method,generator_output_electricity,...,substation,location,cables,circuits,line,layer,osm_id_right,geometry,installed_capacity,energy_source
23,5246574100,generator,NaN,NaN,Trạm cấp xăng An Bình - CA.TPCT,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (105.78503 10.01066),NaN,diesel


In [39]:
# Step 1: 对 filtered_gdf 的 energy_source 进行对应
# 创建一个映射字典，将 energy_source 映射到对应的 INSTALLED CAPACITY (MW)
energy_source_mapping = {
    "hydro": "Hydro",
    "coal": "Coal",
    "oil": "Other Fossil",
    "diesel": "Other Fossil",
    "gas, gas;oil": "Gas",
    "waste": "Bioenergy",
    "biofuel": "Bioenergy", 
    "biogas": "Bioenergy",
    "wind": "Wind",
    "solar": "Solar"
}

# 根据 energy_source 列来映射到 energy_source_stats 列
gens_gdf["energy_source"] = gens_gdf["energy_source"].map(energy_source_mapping)

In [45]:
# 步骤 4：拆分 GeoDataFrame
condition_1 = gens_gdf["installed_capacity"].notnull() & gens_gdf["energy_source"].notnull()
condition_2 = gens_gdf["installed_capacity"].isnull() & gens_gdf["energy_source"].notnull()
condition_3 = gens_gdf["installed_capacity"].notnull() & gens_gdf["energy_source"].isnull()
condition_4 = gens_gdf["installed_capacity"].isnull() & gens_gdf["energy_source"].isnull()

gdf_1 = gens_gdf[condition_1]
gdf_2 = gens_gdf[condition_2]
gdf_3 = gens_gdf[condition_3]
gdf_4 = gens_gdf[condition_4]

print(f"GeoDataFrame 1 (not null installed_capacity and energy_source): {len(gdf_1)} rows")
print(f"GeoDataFrame 2 (null installed_capacity, not null energy_source): {len(gdf_2)} rows")
print(f"GeoDataFrame 3 (not null installed_capacity, null energy_source): {len(gdf_3)} rows")
print(f"GeoDataFrame 4 (null installed_capacity and energy_source): {len(gdf_4)} rows")

GeoDataFrame 1 (not null installed_capacity and energy_source): 539 rows
GeoDataFrame 2 (null installed_capacity, not null energy_source): 1349 rows
GeoDataFrame 3 (not null installed_capacity, null energy_source): 7 rows
GeoDataFrame 4 (null installed_capacity and energy_source): 13 rows


In [46]:
indices_to_remove = pd.concat([gdf_3, gdf_4]).index
filtered_gdf = gens_gdf.drop(indices_to_remove)

In [47]:
print(len(gens_gdf))
print(len(filtered_gdf))

1908
1888


In [48]:
df = pd.read_csv('../data/yearly_full_release_long_format.csv')
installed_capacity_df = df[
    (df["Area"] == "Viet Nam") &
    (df["Year"] == 2023) &
    (df["Category"] == "Capacity") &
    (df["Subcategory"] == "Fuel")
]

installed_capacity_df['Value_MW'] = installed_capacity_df['Value'] * 1000
installed_capacity_df

/tmp/ipykernel_4030750/4214324864.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  installed_capacity_df['Value_MW'] = installed_capacity_df['Value'] * 1000


,Area,Country code,Year,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change,Value_MW
335098,Viet Nam,VNM,2023,Country,Asia,Asia,0.0,0.0,0.0,0.0,1.0,Capacity,Fuel,Bioenergy,GW,0.38,0.00,0.00,380.0
335099,Viet Nam,VNM,2023,Country,Asia,Asia,0.0,0.0,0.0,0.0,1.0,Capacity,Fuel,Coal,GW,27.24,2.63,10.69,27240.0
335100,Viet Nam,VNM,2023,Country,Asia,Asia,0.0,0.0,0.0,0.0,1.0,Capacity,Fuel,Gas,GW,8.15,0.00,0.00,8150.0
335101,Viet Nam,VNM,2023,Country,Asia,Asia,0.0,0.0,0.0,0.0,1.0,Capacity,Fuel,Hydro,GW,22.64,0.11,0.49,22640.0
335102,Viet Nam,VNM,2023,Country,Asia,Asia,0.0,0.0,0.0,0.0,1.0,Capacity,Fuel,Nuclear,GW,NaN,NaN,NaN,NaN
335103,Viet Nam,VNM,2023,Country,Asia,Asia,0.0,0.0,0.0,0.0,1.0,Capacity,Fuel,Other Fossil,GW,NaN,NaN,NaN,NaN
335104,Viet Nam,VNM,2023,Country,Asia,Asia,0.0,0.0,0.0,0.0,1.0,Capacity,Fuel,Solar,GW,17.08,0.38,2.28,17080.0
335105,Viet Nam,VNM,2023,Country,Asia,Asia,0.0,0.0,0.0,0.0,1.0,Capacity,Fuel,Wind,GW,5.89,0.82,16.17,5890.0


In [49]:
gdf_1 = filtered_gdf[condition_1]
gdf_2 = filtered_gdf[condition_2]

/scistor/ivm/mye500/miniconda3/envs/py310/lib/python3.10/site-packages/geopandas/geodataframe.py:1415: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)
/scistor/ivm/mye500/miniconda3/envs/py310/lib/python3.10/site-packages/geopandas/geodataframe.py:1415: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)


In [51]:
# Step 2: 根据 energy_source_stats 分类计算 gdf_1 中的总值
gdf_1_grouped = gdf_1.groupby("energy_source")["installed_capacity"].sum()


In [52]:
gdf_1_grouped

energy_source
Coal     25711.200
Hydro    21108.758
Solar     5432.222
Wind       975.293
Name: installed_capacity, dtype: float64

In [53]:
# Step 3: 从 Excel 的 installed_capacity 减去 gdf_1 的分类总值
remaining_capacity = {}
for _, row in installed_capacity_df.iterrows():
    category = row["Variable"]
    if category in gdf_1_grouped:
        remaining_capacity[category] = row['Value_MW'] - gdf_1_grouped[category]
    else:
        remaining_capacity[category] = row['Value_MW']

In [54]:
remaining_capacity

{'Bioenergy': 380.0,
 'Coal': 1528.7999999999993,
 'Gas': 8150.0,
 'Hydro': 1531.2419999999984,
 'Nuclear': nan,
 'Other Fossil': nan,
 'Solar': 11647.778,
 'Wind': 4914.707}

In [55]:
# 计算 gdf_2 中每个 energy_source 的 installed_capacity 总和

# 定义一个函数，将剩余容量按比例分配到 gdf_2
def allocate_remaining_capacity(row, remaining_capacity, gdf_2_counts):
    stats = row["energy_source"]
    if stats in remaining_capacity and stats in gdf_2_counts:
        return remaining_capacity[stats] / gdf_2_counts[stats]
    return 0  # 如果没有对应的剩余值或分组总和，分配为 0

# 计算 gdf_2 中每类 energy_source 的行数
gdf_2_counts = gdf_2["energy_source"].value_counts()

# 创建一个新列，将剩余容量分配到 gdf_2 的每一行
gdf_2["installed_capacity"] = gdf_2.apply(
    allocate_remaining_capacity, axis=1, 
    remaining_capacity=remaining_capacity, 
    gdf_2_counts=gdf_2_counts
)

gdf_2

/scistor/ivm/mye500/miniconda3/envs/py310/lib/python3.10/site-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,osm_id,power,voltage,utility,name,frequency,plant_output_electricity,plant_source,plant_method,generator_output_electricity,...,substation,location,cables,circuits,line,layer,osm_id_right,geometry,installed_capacity,energy_source
0,2495831951,generator,NaN,NaN,Nhà máy nhiệt điện HẢi Phòng,NaN,NaN,NaN,NaN,yes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (106.65356 20.84855),764.400000,Coal
22,5207569281,generator,NaN,NaN,Trạm biến áp 110/220v,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (105.76793 10.02529),126.666667,Bioenergy
23,5246574100,generator,NaN,NaN,Trạm cấp xăng An Bình - CA.TPCT,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (105.78503 10.01066),NaN,Other Fossil
24,5246576900,generator,NaN,NaN,Tram xang dau QL1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (105.74960 9.99618),126.666667,Bioenergy
25,5573888849,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (108.92361 10.54565),5.256371,Wind
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1903,1308761089,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (107.67483 13.30744, 107.67484 13.3...",41.013303,Solar
1904,1308761090,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (107.67319 13.30557, 107.67174 13.3...",41.013303,Solar
1905,1308761091,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (107.67158 13.30556, 107.67011 13.3...",41.013303,Solar
1906,1308761092,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (107.66668 13.30942, 107.66509 13.3...",41.013303,Solar


In [56]:
gens_gdf_update = pd.concat([gdf_1, gdf_2], ignore_index=True)

In [57]:
gens_gdf_update

,osm_id,power,voltage,utility,name,frequency,plant_output_electricity,plant_source,plant_method,generator_output_electricity,...,substation,location,cables,circuits,line,layer,osm_id_right,geometry,installed_capacity,energy_source
0,4037744488,plant,NaN,NaN,Nhà máy thủy điện Ialy,NaN,720 MW,hydro,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (107.79546 14.22273),720.000000,Hydro
1,4195284426,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500 kW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (108.68039 11.20209),1.500000,Wind
2,4195284427,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500 kW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (108.67951 11.20397),1.500000,Wind
3,4195284428,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500 kW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (108.68835 11.20445),1.500000,Wind
4,4195284429,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500 kW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (108.67859 11.20585),1.500000,Wind
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1883,1308761089,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (107.67483 13.30744, 107.67484 13.3...",41.013303,Solar
1884,1308761090,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (107.67319 13.30557, 107.67174 13.3...",41.013303,Solar
1885,1308761091,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (107.67158 13.30556, 107.67011 13.3...",41.013303,Solar
1886,1308761092,generator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (107.66668 13.30942, 107.66509 13.3...",41.013303,Solar


In [58]:
gens_gdf_update.to_file("../data/osm/vietnam-latest_plants_and_generators_update_capacity.gpkg", driver="GPKG")